# Qiskit local state vector simulation

The best way to do this is with Qiskit - Aer. It uses ```AerSimulator```, which is optimized for performance and allows you to simulate circuits with many qubits locally.\
You must explicitly tell the simulator to save the state vector using ```qc.save_statevector()``` before running the simulation; otherwise, it will discard the complex vector to save memory.

In [10]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import array_to_latex

# 1. Create a simple quantum circuit (Bell State)
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# IMPORTANT: You must save the statevector before measuring
# If you add measurements *before* this, the state will collapse!
qc.save_statevector() 

# 2. Set up the local simulator
# method='statevector' forces a full state vector simulation
simulator = AerSimulator(method='statevector')

# 3. Transpile the circuit for the simulator
qc_transpiled = transpile(qc, simulator)

# 4. Run the simulation
result = simulator.run(qc_transpiled).result()

# 5. Extract the state vector
statevector = result.get_statevector(qc_transpiled)

print(f"State Vector:\n {statevector}")

# Optional: Visualize as LaTeX (requires matplotlib and latex installed)
print(array_to_latex(statevector, source=True))

State Vector:
 Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))


\begin{bmatrix}
\frac{\sqrt{2}}{2} & 0 & 0 & \frac{\sqrt{2}}{2}  \\
 \end{bmatrix}



Visualize the state vector in LaTeX
$$\begin{bmatrix}
\frac{\sqrt{2}}{2} & 0 & 0 & \frac{\sqrt{2}}{2}  \\
 \end{bmatrix}$$

# Variational Quantum Algorithms
Run this locally -- without IBM hardware -- use the Primitives ```StatevectorEstimator``` to compute the math and impliment a classical optimizers (e.g. SPSA, COBYLA) to update the parameters.

In [22]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import COBYLA, SPSA

# Define the Ansatz
# We want to tune 'theta' to a specific value
theta = Parameter('theta')
qc = QuantumCircuit(1)
qc.rx(theta, 0)  # Rotate X-axis by theta

# 2. Define the "Hamiltonian" (The Cost Function Target)
# We want to minimize the expectation value of the Z operator.
# The minimum value for Z is -1 (state |1>)
hamiltonian = SparsePauliOp.from_list([("Z", 1)])

# 3. Setup the Local Estimator
# This replaces the quantum computer. It calculates exact expectation values locally.
estimator = StatevectorEstimator()

# 4. Define the Cost Function
# This function takes parameters, runs the "circuit", and returns a score (cost).
def cost_function(params):
    # Create a "Probability Unified Bloch" (PUB) - the modern Qiskit input format
    # Tuple format: (circuit, observables, parameter_values)
    pub = (qc, hamiltonian, params)
    
    # Run the job locally
    job = estimator.run([pub])
    result = job.result()
    
    # Extract the expectation value (evs)
    # We return the float value for the optimizer to minimize
    cost = result[0].data.evs
    return cost

# 5. Run the Classical Optimization
# We start with a random guess for theta
initial_point = np.random.random(1)

# Initialize the optimizer
# COBYLA is great for noise-free simulation. 
# SPSA is better if you simulate noise, but COBYLA is faster here.
optimizer = COBYLA(maxiter=100) 

print(f"Initial Parameter: {initial_point[0]:.4f}")

# The optimizer minimizes our cost_function
result = optimizer.minimize(fun=cost_function, x0=initial_point)

print("--- Optimization Complete ---")
print(f"Optimal Parameter (Theta): {result.x[0]:.4f}")
print(f"Minimum Cost Found: {result.fun:.4f}")
print(f"Target Cost was: -1.0")

ImportError: cannot import name 'BaseSampler' from 'qiskit.primitives' (/opt/anaconda3/lib/python3.12/site-packages/qiskit/primitives/__init__.py)

In [3]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

# IMPORT SCIPY instead of qiskit_algorithms
from scipy.optimize import minimize

# 1. Define Circuit (Same as before)
theta = Parameter('theta')
qc = QuantumCircuit(1)
qc.rx(theta, 0)

# 2. Define Hamiltonian (Same as before)
hamiltonian = SparsePauliOp.from_list([("Z", 1)])

# 3. Setup Local Estimator
estimator = StatevectorEstimator()

# 4. Define Cost Function
# Scipy requires the function to accept a list of floats (params) 
# and return a single float (cost)
def cost_function(params):
    pub = (qc, hamiltonian, params)
    job = estimator.run([pub])
    result = job.result()
    # Scipy expects a simple float, not a complex object
    expval = result[0].data.evs
    print(f"expval:{expval}")
    return expval

# 5. Run Optimization using Scipy
initial_point = np.random.random(1)
print(f"Initial Parameter: {initial_point[0]:.4f}")

# 'method' can be 'COBYLA', 'BFGS', 'Nelder-Mead', etc.
result = minimize(cost_function, initial_point, method='COBYLA')

print("--- Optimization Complete (via Scipy) ---")
print(f"Optimal Parameter: {result.x[0]:.4f}")
print(f"Minimum Cost: {result.fun:.4f}")
print(f"Success: {result.success}")

Initial Parameter: 0.0240
expval:0.9997112720381195
expval:0.5199269543049846
expval:-0.43787580745015575
expval:-0.6352700078963038
expval:0.9666070157288057
expval:-0.9930975712033697
expval:-0.43787580745015575
expval:-0.9277574839181277
expval:-0.9332062776014333
expval:-0.9998457962075955
expval:-0.9966038925211935
expval:-0.9977185721197384
expval:-0.9999723364409624
expval:-0.9994739265145584
expval:-0.999996718687563
expval:-0.9999211020956229
expval:-0.999997027501065
expval:-0.9999878069312828
expval:-0.9999989657370058
expval:-0.9999999039740642
expval:-0.9999998422113023
expval:-0.9999995598554775
expval:-0.9999999822833717
expval:-0.9999999980926804
expval:-0.9999999514019897
expval:-0.9999999992689569
expval:-0.9999999904452334
--- Optimization Complete (via Scipy) ---
Optimal Parameter: 3.1416
Minimum Cost: -1.0000
Success: True


## Plot the parameter landscape